# Visualize Feature PTH Bounding Boxes

This notebook loads a feature `.pth` file, reads `image` and `bbox`, draws bounding boxes, and saves annotated images.

In [2]:
from pathlib import Path
from typing import Any, Iterable

import torch
from PIL import Image, ImageDraw

In [3]:
# Input/output paths
pth_path = Path("/mnt/abka03/Projects/xl-vlms/outputs/run_test_tiger_qwen7b/features/save_hidden_states_mean_qwen2_patched_image_cat_token_of_interest_concept_generation_split_train_blue.pth")
out_dir = Path("/mnt/abka03/Projects/xl-vlms/outputs/run_test_tiger_qwen7b/features/vis_blue_bbox")
max_items = 5  # 0 means all

In [4]:
def _to_float_list(x: Any):
    if x is None:
        return None
    if isinstance(x, (list, tuple)) and len(x) == 4:
        try:
            return [float(v) for v in x]
        except Exception:
            return None
    return None


def _safe_get(seq: Any, idx: int, default=None):
    try:
        return seq[idx]
    except Exception:
        return default


def _iter_indices(total: int, max_items: int) -> Iterable[int]:
    if max_items <= 0:
        yield from range(total)
    else:
        yield from range(min(total, max_items))


def visualize_pth(pth_path: Path, out_dir: Path, max_items: int = 0):
    data = torch.load(pth_path, map_location="cpu")
    if not isinstance(data, dict):
        raise ValueError(f"Expected dict in {pth_path}, got {type(data)}")

    images = data.get("image", [])
    bboxes = data.get("bbox", [])
    preds = data.get("model_predictions", [])
    concepts = data.get("concept", [])
    is_concepts = data.get("is_concept", [])

    if not isinstance(images, list) or not isinstance(bboxes, list):
        raise ValueError("Expected list fields: image and bbox")

    out_dir.mkdir(parents=True, exist_ok=True)

    saved = 0
    missing = 0

    for i in _iter_indices(len(images), max_items):
        img_path = Path(str(images[i]))
        bbox = _to_float_list(_safe_get(bboxes, i))
        pred = _safe_get(preds, i, "")
        concept = _safe_get(concepts, i, "")
        is_concept = _safe_get(is_concepts, i, None)

        if not img_path.exists():
            missing += 1
            continue

        img = Image.open(img_path).convert("RGB")
        draw = ImageDraw.Draw(img)

        if bbox is not None:
            x, y, w, h = bbox
            x2 = x + w
            y2 = y + h
            draw.rectangle([(x, y), (x2, y2)], outline=(255, 0, 0), width=3)

        meta = f"idx={i}"
        if concept:
            meta += f" | concept={concept}"
        if pred:
            meta += f" | pred={pred}"
        if is_concept is not None:
            meta += f" | is_concept={is_concept}"

        draw.rectangle([(0, 0), (img.width, 24)], fill=(0, 0, 0))
        draw.text((6, 4), meta, fill=(255, 255, 255))

        out_name = out_dir / f"{i:05d}_{img_path.stem}_bbox.jpg"
        img.save(out_name, quality=92)
        saved += 1

    return saved, missing

In [5]:
saved, missing = visualize_pth(pth_path, out_dir, max_items=max_items)
print(f"Saved {saved} annotated images to: {out_dir}")
if missing:
    print(f"Skipped {missing} entries with missing image files")

Saved 5 annotated images to: /mnt/abka03/Projects/xl-vlms/outputs/run_test_tiger_qwen7b/features/vis_blue_bbox
